In [1]:
import nibabel as nib
import numpy as np
import os
from tqdm import tqdm

In [2]:
def load_nifti_file(file_path):
    """Load a NIfTI file and return the image data as a numpy array."""
    img = nib.load(file_path)
    data = img.get_fdata()
    return data



def save_nifti_min_size(np_array, file_path, affine=None, dtype=np.float32):
    """
    Save a NumPy array as a compressed NIfTI file with minimum size.

    Parameters
    ----------
    np_array : np.ndarray
        The image data to save.
    file_path : str
        Output path for .nii.gz file (recommended).
    affine : np.ndarray, optional
        4×4 affine matrix. If None, identity is used.
    dtype : np.dtype
        Data type to save as (e.g., np.float32, np.uint8).
    """

    # Use identity affine if none provided
    if affine is None:
        affine = np.eye(4)

    # Convert array dtype to reduce size
    np_array = np.asarray(np_array).astype(dtype)

    # Create and save compressed NIfTI
    nifti_img = nib.Nifti1Image(np_array, affine)

    if not file_path.endswith(".gz"):
        file_path += ".gz"  # enforce gzip compression

    nib.save(nifti_img, file_path)
    print(f"Saved compressed NIfTI file (dtype={dtype}) to: {file_path}")



def read_json(file_path):
    """Read a JSON file and return its content as a dictionary."""
    import json
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def iterate_over_all_files(root_dir):
    data_directories_dir = os.path.join(root_dir, "dataset_directories.json")
    dict_dataset = read_json(data_directories_dir)
    
    center_1_root = os.path.join(root_dir, "Masih-SUV")
    center_1_dicts = dict_dataset.get("Masih-SUV", [])
    for item in tqdm(center_1_dicts, desc="Processing Center 1 Files"):
        sample_dir_name, sample_info = list(item.items())[0]
        img_name = sample_info['image']
        img_dir = os.path.join(center_1_root, sample_dir_name, img_name.replace('.nii', '_prep_img.nii.gz'))
        seg_name = sample_info['segmentation']
        seg_dir = os.path.join(center_1_root, sample_dir_name, seg_name.replace('.nrrd', '_prep_seg.nii.gz'))
        img_data = load_nifti_file(img_dir)
        seg_data = load_nifti_file(seg_dir)
        hl_label = 0 if sample_info['Binary Label'] == 'NHL' else 1
        yield seg_data, hl_label, img_data
    
def find_max_dimensions(root_dir):
    max_dims = [0, 0, 0]
    for seg_data, _, _ in iterate_over_all_files(root_dir):
        current_dims = seg_data.shape
        for i in range(3):
            if current_dims[i] > max_dims[i]:
                max_dims[i] = current_dims[i]
    return max_dims



In [3]:
root_dir = "/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/new_dataset"

In [7]:
max_size = find_max_dimensions(root_dir)
print("Maximum dimensions across all segmentation files:", max_size)

Processing Center 1 Files: 100%|██████████| 151/151 [00:20<00:00,  7.52it/s]

Maximum dimensions across all segmentation files: [224, 224, 371]


In [16]:
def nodes_heatmap(root_dir, max_size=[224,224,371]):
    hodgkin_heatmap = np.zeros(max_size)
    non_hodgkin_heatmap = np.zeros(max_size)
    image_heatmap = np.zeros(max_size)
    hodgkin_num, non_hodgkin_num = 0, 0
    for seg_data, hl_label, img_data in iterate_over_all_files(root_dir):
        if hl_label == 1:
            hodgkin_num += 1
            hodgkin_heatmap[:seg_data.shape[0], :seg_data.shape[1], :seg_data.shape[2]] += (seg_data > 0).astype(np.uint)
        else:
            non_hodgkin_num += 1
            non_hodgkin_heatmap[:seg_data.shape[0], :seg_data.shape[1], :seg_data.shape[2]] += (seg_data > 0).astype(np.uint8)
        image_heatmap[:img_data.shape[0], :img_data.shape[1], :img_data.shape[2]] += img_data
    hodgkin_heatmap /= hodgkin_num
    non_hodgkin_heatmap /= non_hodgkin_num
    image_heatmap /= (hodgkin_num + non_hodgkin_num)
    return hodgkin_heatmap, non_hodgkin_heatmap, image_heatmap

hodgkin_heatmap, non_hodgkin_heatmap, image_heatmap = nodes_heatmap(root_dir, max_size)

Processing Center 1 Files: 100%|██████████| 151/151 [01:43<00:00,  1.45it/s]


In [19]:
os.makedirs("location_heatmaps", exist_ok=True)
save_nifti_min_size(hodgkin_heatmap, "location_heatmaps/hodgkin_heatmap.nii.gz")
save_nifti_min_size(non_hodgkin_heatmap, "location_heatmaps/non_hodgkin_heatmap.nii.gz")
save_nifti_min_size(image_heatmap, "location_heatmaps/image_heatmap.nii.gz")

Saved compressed NIfTI file (dtype=<class 'numpy.float32'>) to: location_heatmaps/hodgkin_heatmap.nii.gz
Saved compressed NIfTI file (dtype=<class 'numpy.float32'>) to: location_heatmaps/non_hodgkin_heatmap.nii.gz
Saved compressed NIfTI file (dtype=<class 'numpy.float32'>) to: location_heatmaps/image_heatmap.nii.gz


# find the heatmaps with registration

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import SimpleITK as sitk


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)


def iterate_case_paths(root_dir):
    """
    Yields:
      sample_dir_name, img_path, seg_path, hl_label (1=Hodgkin, 0=Non-Hodgkin)
    """
    data_directories_path = os.path.join(root_dir, "dataset_directories.json")
    dict_dataset = read_json(data_directories_path)

    center_1_root = os.path.join(root_dir, "Masih-SUV")
    center_1_dicts = dict_dataset.get("Masih-SUV", [])

    for item in center_1_dicts:
        sample_dir_name, sample_info = list(item.items())[0]

        img_name = sample_info["image"]
        img_path = os.path.join(
            center_1_root,
            sample_dir_name,
            img_name.replace(".nii", "_prep_img.nii.gz"),
        )

        seg_name = sample_info["segmentation"]
        seg_path = os.path.join(
            center_1_root,
            sample_dir_name,
            seg_name.replace(".nrrd", "_prep_seg.nii.gz"),
        )

        hl_label = 0 if sample_info["Binary Label"] == "NHL" else 1
        yield sample_dir_name, img_path, seg_path, hl_label


def register_affine(fixed_img, moving_img, shrink_factors=(4, 2, 1), smoothing_sigmas=(2, 1, 0)):
    """
    Minimal robust affine registration: Mattes MI + multi-resolution GD.
    Returns: affine_transform (sitk.Transform)
    """
    # Initialize around centers (good default)
    initial = sitk.CenteredTransformInitializer(
        fixed_img,
        moving_img,
        sitk.AffineTransform(3),
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.02)  # keep small for speed; increase for stability
    reg.SetInterpolator(sitk.sitkLinear)

    reg.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=200,
        convergenceMinimumValue=1e-6,
        convergenceWindowSize=10,
    )
    reg.SetOptimizerScalesFromPhysicalShift()

    reg.SetShrinkFactorsPerLevel(shrinkFactors=shrink_factors)
    reg.SetSmoothingSigmasPerLevel(smoothingSigmas=smoothing_sigmas)
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    reg.SetInitialTransform(initial, inPlace=False)

    final_tfm = reg.Execute(fixed_img, moving_img)
    return final_tfm


def resample_to_fixed(moving, fixed, transform, is_label=False):
    """
    Resample moving image into fixed space.
    """
    interp = sitk.sitkNearestNeighbor if is_label else sitk.sitkLinear
    default_value = 0
    return sitk.Resample(moving, fixed, transform, interp, default_value, moving.GetPixelID())


def build_registered_heatmaps(
    root_dir: str,
    fixed_sample_dir_name: str,
    out_dir: str,
    save_mean_pet: bool = True,
):
    os.makedirs(out_dir, exist_ok=True)

    # --- find fixed paths
    fixed_img_path = None
    fixed_seg_path = None
    for sd, img_p, seg_p, _ in iterate_case_paths(root_dir):
        if sd == fixed_sample_dir_name:
            fixed_img_path = img_p
            fixed_seg_path = seg_p
            break
    if fixed_img_path is None:
        raise ValueError(f"Could not find fixed_sample_dir_name='{fixed_sample_dir_name}' in dataset_directories.json")

    # --- read fixed (float)
    fixed_img = sitk.ReadImage(fixed_img_path)
    fixed_img = sitk.Cast(fixed_img, sitk.sitkFloat32)

    # --- accumulators in fixed space (float32)
    fixed_size = fixed_img.GetSize()  # (x,y,z)
    hodgkin_acc = sitk.Image(fixed_size, sitk.sitkFloat32)
    nonhodgkin_acc = sitk.Image(fixed_size, sitk.sitkFloat32)
    mean_pet_acc = sitk.Image(fixed_size, sitk.sitkFloat32)

    # copy geometry
    for im in (hodgkin_acc, nonhodgkin_acc, mean_pet_acc):
        im.CopyInformation(fixed_img)

    hodgkin_n = 0
    nonhodgkin_n = 0
    total_n = 0

    # --- loop all cases, register to fixed, accumulate
    for sd, img_path, seg_path, hl_label in tqdm(list(iterate_case_paths(root_dir)), desc="Registering & accumulating"):
        moving_img = sitk.ReadImage(img_path)
        moving_img = sitk.Cast(moving_img, sitk.sitkFloat32)

        moving_seg = sitk.ReadImage(seg_path)
        # ensure binary label (tumor>0)
        moving_seg = sitk.Cast(moving_seg > 0, sitk.sitkUInt8)

        # transform (affine)
        tfm = register_affine(fixed_img, moving_img)

        # resample both into fixed space
        warped_img = resample_to_fixed(moving_img, fixed_img, tfm, is_label=False)
        warped_seg = resample_to_fixed(moving_seg, fixed_img, tfm, is_label=True)
        warped_seg = sitk.Cast(warped_seg > 0, sitk.sitkFloat32)

        # accumulate
        mean_pet_acc = mean_pet_acc + warped_img

        if hl_label == 1:
            hodgkin_acc = hodgkin_acc + warped_seg
            hodgkin_n += 1
        else:
            nonhodgkin_acc = nonhodgkin_acc + warped_seg
            nonhodgkin_n += 1

        total_n += 1

    # --- normalize to probabilities / mean image
    if hodgkin_n > 0:
        hodgkin_heatmap = hodgkin_acc / float(hodgkin_n)
    else:
        hodgkin_heatmap = hodgkin_acc  # all zeros

    if nonhodgkin_n > 0:
        nonhodgkin_heatmap = nonhodgkin_acc / float(nonhodgkin_n)
    else:
        nonhodgkin_heatmap = nonhodgkin_acc  # all zeros

    mean_pet = mean_pet_acc / float(max(total_n, 1))

    # --- save outputs
    if os.path.exists(out_dir) is False:
        os.makedirs(out_dir)
        
    fixed_out = os.path.join(out_dir, "fixed_pet.nii.gz")
    h_out = os.path.join(out_dir, "hodgkin_heatmap.nii.gz")
    nh_out = os.path.join(out_dir, "non_hodgkin_heatmap.nii.gz")

    sitk.WriteImage(fixed_img, fixed_out)
    sitk.WriteImage(hodgkin_heatmap, h_out)
    sitk.WriteImage(nonhodgkin_heatmap, nh_out)

    if save_mean_pet:
        mean_out = os.path.join(out_dir, "mean_pet.nii.gz")
        sitk.WriteImage(mean_pet, mean_out)

    return {
        "fixed_pet": fixed_out,
        "hodgkin_heatmap": h_out,
        "non_hodgkin_heatmap": nh_out,
        "mean_pet": os.path.join(out_dir, "mean_pet.nii.gz") if save_mean_pet else None,
        "counts": {"hodgkin": hodgkin_n, "non_hodgkin": nonhodgkin_n, "total": total_n},
        "fixed_case": fixed_sample_dir_name,
        "fixed_img_path": fixed_img_path,
        "fixed_seg_path": fixed_seg_path,
    }


In [ ]:
out = build_registered_heatmaps(
    root_dir=root_dir,
    fixed_sample_dir_name="ALI HOSEYNI FAEZEH 9705000138",   # <-- pick any folder name in Masih-SUV list
    out_dir=os.path.join('location_heatmaps', "registered_heatmaps"),
    save_mean_pet=True,
)

print(out)
